<a href="https://colab.research.google.com/github/j019/Practical-Machine-Learning/blob/main/Day17/Pytorch_multiclass_mobile_price.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd

In [2]:
df = pd.read_csv("/content/mobile_price_prediction.csv")

In [3]:
df.shape

(2000, 21)

In [4]:
df.columns

Index(['battery_power', 'blue', 'clock_speed', 'dual_sim', 'fc', 'four_g',
       'int_memory', 'm_dep', 'mobile_wt', 'n_cores', 'pc', 'px_height',
       'px_width', 'ram', 'sc_h', 'sc_w', 'talk_time', 'three_g',
       'touch_screen', 'wifi', 'price_range'],
      dtype='object')

In [5]:
X = df.drop('price_range',axis=1)
y = df['price_range']

In [6]:
y.value_counts(normalize=True)

,proportion
price_range,
1,0.25
2,0.25
3,0.25
0,0.25


In [7]:
import torch
import torch.nn as nn
import torch.optim as optim

In [8]:
X_32 = torch.tensor(X.values, dtype=torch.float32)
# To have output in integer format keep the datatype of y as long
y_32 = torch.tensor(y.values, dtype=torch.long)

In [9]:
input_size = X_32.shape[1]

In [10]:
# define the model
model = nn.Sequential(
    nn.Linear(input_size, 12), # HL1 all columns in X as inputs and 12 neuron so 12 output
    nn.ReLU(), # activation function layer for HL1
    nn.Linear(12, 8), # HL2 12 inputs and 8 neuron so 8 output
    nn.ReLU(),# activation function layer for HL2
    nn.Linear(8, 4),# output layer 9 inputs and 4 neuron so 4 output
    # No need of softmax / sigmoid layer
    # nn.Softmax()# activation function layer for output layer
)
print(model)

Sequential(
  (0): Linear(in_features=20, out_features=12, bias=True)
  (1): ReLU()
  (2): Linear(in_features=12, out_features=8, bias=True)
  (3): ReLU()
  (4): Linear(in_features=8, out_features=4, bias=True)
)


In [11]:
# Define loss function
# loss_fn   = nn.CTCLoss( )  # binary cross entropy
loss_fn = nn.CrossEntropyLoss()
# difference in loss function
# https://stackoverflow.com/questions/61437961/is-crossentropy-loss-of-pytorch-different-than-categorical-crossentropy-of-ker

In [12]:
# define optimizer for weights and bias of model (model.parameters)
optimizer = optim.Adam(model.parameters(), lr=0.001)

In [13]:
# define epoch and batch size
n_epochs = 100
batch_size = 10

In [14]:
for epoch in range(n_epochs):
    for i in range(0, len(X_32), batch_size):
        # forward pass
        Xbatch = X_32[i:i+batch_size]
        y_pred = model(Xbatch)
        ybatch = y_32[i:i+batch_size]
        loss = loss_fn(y_pred, ybatch)
        # back propagation
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
    print(f'Finished epoch {epoch}, latest loss {loss}')

Finished epoch 0, latest loss 1.5522334575653076
Finished epoch 1, latest loss 0.5862724781036377
Finished epoch 2, latest loss 0.537811279296875
Finished epoch 3, latest loss 0.5291123390197754
Finished epoch 4, latest loss 0.6072837114334106
Finished epoch 5, latest loss 0.6374215483665466
Finished epoch 6, latest loss 0.6252354979515076
Finished epoch 7, latest loss 0.5750784277915955
Finished epoch 8, latest loss 0.5171844959259033
Finished epoch 9, latest loss 0.4918191432952881
Finished epoch 10, latest loss 0.511167585849762
Finished epoch 11, latest loss 0.49419569969177246
Finished epoch 12, latest loss 0.5780415534973145
Finished epoch 13, latest loss 0.5108902454376221
Finished epoch 14, latest loss 0.488553524017334
Finished epoch 15, latest loss 0.5386035442352295
Finished epoch 16, latest loss 0.5336922407150269
Finished epoch 17, latest loss 0.5445046424865723
Finished epoch 18, latest loss 0.5152920484542847
Finished epoch 19, latest loss 0.5098387002944946
Finished epo

In [15]:
# compute accuracy (no_grad is optional)
with torch.no_grad():
    y_pred = model(X_32)
accuracy = (y_pred.argmax(dim=1) == y_32).float().mean() #2.5 round to 2, 2.3 to 2, 2.9 to 3
print(f"Accuracy {accuracy}")

Accuracy 0.7894999980926514


In [16]:
y_pred.argmax(dim=1)

tensor([2, 2, 2,  ..., 3, 0, 3])

In [17]:
X_32

tensor([[8.4200e+02, 0.0000e+00, 2.2000e+00,  ..., 0.0000e+00, 0.0000e+00,
         1.0000e+00],
        [1.0210e+03, 1.0000e+00, 5.0000e-01,  ..., 1.0000e+00, 1.0000e+00,
         0.0000e+00],
        [5.6300e+02, 1.0000e+00, 5.0000e-01,  ..., 1.0000e+00, 1.0000e+00,
         0.0000e+00],
        ...,
        [1.9110e+03, 0.0000e+00, 9.0000e-01,  ..., 1.0000e+00, 1.0000e+00,
         0.0000e+00],
        [1.5120e+03, 0.0000e+00, 9.0000e-01,  ..., 1.0000e+00, 1.0000e+00,
         1.0000e+00],
        [5.1000e+02, 1.0000e+00, 2.0000e+00,  ..., 1.0000e+00, 1.0000e+00,
         1.0000e+00]])

# Model size


Ref: https://discuss.pytorch.org/t/finding-model-size/130275/2

In [18]:
param_size = 0
for param in model.parameters():
    param_size += param.nelement() * param.element_size()
buffer_size = 0
for buffer in model.buffers():
    buffer_size += buffer.nelement() * buffer.element_size()

size_all_mb = (param_size + buffer_size) / 1024**2
print('model size: {:.3f}MB'.format(size_all_mb))

model size: 0.001MB
